# Week 7: Build a Text Classifier

**Grade band:** 6 to 8  |  **Duration:** 60 minutes  |  **Platform:** JupyterLite (browser, no account) or Google Colab

**How to use this notebook:** run each cell from top to bottom with Shift + Enter. Read the text, run the code, then complete the challenge cells marked **YOUR TURN**. Save your work at the end of the session (File > Download) so it can be uploaded to your portfolio.

## Hook: Spam or not spam

"CONGRATULATIONS you won a FREE prize click now." Your email app sorted that into spam before you saw it. Nobody wrote a rule for that exact sentence. Today you will build the same kind of model, trained on examples.

In [ ]:
# Setup: run this cell first.
import pandas as pd
import matplotlib.pyplot as plt

# If a data file is not found next to this notebook (for example on Google Colab),
# it is loaded from the Wize data folder online instead. Replace this URL after publishing.
DATA_URL = "https://raw.githubusercontent.com/wizeacademy/ml-ai-6-8/main/notebooks/data/"

def load(name):
    """Load a Wize dataset by file name, from the local data folder or from the web."""
    try:
        return pd.read_csv("data/" + name)
    except Exception:
        return pd.read_csv(DATA_URL + name)

print("Setup complete. pandas and matplotlib are ready.")

## Teach 1: Turning words into numbers

Models only understand numbers. `CountVectorizer` builds a **vocabulary** of every word it sees, then turns each sentence into counts: how many times each word appears. This is called a **bag of words**.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

tiny = ["the game is fun", "the game is boring", "fun fun fun"]
vec = CountVectorizer()
counts = vec.fit_transform(tiny)

print("Vocabulary:", vec.get_feature_names_out())
print(pd.DataFrame(counts.toarray(), columns=vec.get_feature_names_out()))

## Teach 2: Train the classifier

**Naive Bayes** learns how likely each word is in a positive review versus a negative one, then adds up the evidence. It is fast and works well on text.

**Concept checkpoint:** before running, predict which word the model will think is the most "negative" in our reviews.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

reviews = load("game_reviews.csv")
X_text_train, X_text_test, y_train, y_test = train_test_split(
    reviews["review"], reviews["label"], test_size=0.25, random_state=3)

vec = CountVectorizer()
X_train = vec.fit_transform(X_text_train)      # learn vocabulary AND transform
X_test = vec.transform(X_text_test)            # only transform (same vocabulary)

clf = MultinomialNB()
clf.fit(X_train, y_train)
print("Test accuracy:", round(accuracy_score(y_test, clf.predict(X_test)) * 100, 1), "%")

In [ ]:
# Which words carry the most evidence?
import numpy as np
words = vec.get_feature_names_out()
neg_idx = list(clf.classes_).index("negative")
pos_idx = list(clf.classes_).index("positive")
evidence = clf.feature_log_prob_[pos_idx] - clf.feature_log_prob_[neg_idx]
order = np.argsort(evidence)
print("Most negative words:", list(words[order[:8]]))
print("Most positive words:", list(words[order[-8:]]))

## YOUR TURN: Challenge

**Timer suggestion: 25 minutes.**

### Mild
Write three new reviews and ask the model to classify them. Use `vec.transform` then `clf.predict`.

### Medium
Add ten labeled sentences of your own to the training data (five positive, five negative) about a topic you know: a movie, a sport, a food. Retrain and test on three new sentences about that topic.

### Spicy
The bag of words cannot tell "not fun" from "fun". Rebuild the vectorizer with `ngram_range=(1, 2)` so it also counts word pairs. Compare accuracy. Then test "this is not fun at all" on both models.

In [ ]:
# MILD: classify new reviews
new_reviews = [
    # TODO: three sentences
]
# TODO: predictions = clf.predict(vec.transform(new_reviews))
# TODO: print each review with its prediction

In [ ]:
# MEDIUM: add your own training data
my_texts = [
    # TODO: five positive and five negative sentences about your topic
]
my_labels = [
    # TODO: "positive" or "negative" for each, in the same order
]
all_texts = list(reviews["review"]) + my_texts
all_labels = list(reviews["label"]) + my_labels
# TODO: new vectorizer + new MultinomialNB trained on all_texts / all_labels
# TODO: predict three new sentences about your topic

In [ ]:
# SPICY: word pairs (bigrams)
vec_pairs = CountVectorizer(ngram_range=(1, 2))
# TODO: fit_transform on X_text_train, transform X_text_test
# TODO: train clf_pairs and compare test accuracy with clf
# TODO: predict "this is not fun at all" with both models

## Extra activities (if you finish early)

- Print the full vocabulary size. How many words did the model learn from 80 reviews?
- Try a review written in ALL CAPS. Does the model care? (Hint: `CountVectorizer` lowercases by default.)
- **Colab only:** run `from transformers import pipeline; pipeline("sentiment-analysis")("this is not fun at all")` to see a model trained on millions of sentences. Compare it to yours.

## Reflection

- Why does the model need to see examples of both classes?
- What is one thing a bag of words throws away about a sentence?